In [ ]:
!git clone https://github.com/kaizar-rang/yanglab-acne.git
%cd yanglab-acne
!git clone https://github.com/ultralytics/yolov5
%pip install roboflow -q
%pip install -r requirements.txt -q
%pip install -r yolov5/requirements.txt -q
print("\nDone. Restart the runtime now (Runtime -> Restart session), then continue from Cell 2.")

In [ ]:
import os, sys, yaml, getpass
from pathlib import Path

%cd /content/yanglab-acne
os.environ["WANDB_DISABLED"] = "true"

roboflow_key = getpass.getpass("Enter ROBOFLOW_API_KEY: ")
os.environ["ROBOFLOW_API_KEY"] = roboflow_key

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
version = rf.workspace("acne-vulgaris-detection").project("acne04-detection").version(1)
version.download("yolov5", location="data/acne04", overwrite=True)

repo_root = Path.cwd()
config = {
    "train": str(repo_root / "data/acne04/train/images"),
    "val": str(repo_root / "data/acne04/valid/images"),
    "test": str(repo_root / "data/acne04/test/images"),
    "nc": 4,
    "names": ["nodules and cysts", "papules", "pustules", "whitehead and blackhead"]
}
os.makedirs("configs", exist_ok=True)
with open("configs/acne04.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Setup complete. Working directory: {os.getcwd()}")

In [ ]:
!python yolov5/train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data configs/acne04.yaml \
    --weights yolov5s.pt \
    --project outputs/checkpoints \
    --name yolov5_acne \
    --exist-ok

In [ ]:
!python yolov5/detect.py \
    --weights outputs/checkpoints/yolov5_acne/weights/best.pt \
    --source data/acne04/test/images \
    --data configs/acne04.yaml \
    --conf 0.25 \
    --project outputs/predictions \
    --name yolov5_detections \
    --exist-ok

In [ ]:
from google.colab import files

!zip -r yolov5_results.zip outputs/checkpoints/yolov5_acne/
files.download('yolov5_results.zip')

!zip -r yolov5_detections.zip outputs/predictions/yolov5_detections/
files.download('yolov5_detections.zip')